# Gradient Descent - Solutions

In the lecture, we have seen that fitting a model is an optimization problem: we search the parameters that minimize the error of the model.
For linear regression, the minimum can be computed directly (closed-form solution, see the previous exercise). For most other models, e.g., neural networks, this is not possible.
Instead, the minimum is searched iteratively with **gradient descent**.

In this exercise, we implement gradient descent ourselves:
first for a simple function of two variables to understand the mechanics, then to fit the production time model of the laser cutting process from the previous exercise.

Questions in the tasks are meant for thinking - you do not have to write down your answers. Tasks marked with **Experiment** ask you to change your code and to observe the effect.

Tasks marked with **Optional** go one step further. Skip them if you are short on time - they are not needed for the following tasks.

### Contents <a id="contents"></a>
1. [Gradient Descent for Function Minimization](#1-gradient-descent-for-function-minimization)
    * [1.1 Plot Functions](#1-1-plot-functions)
    * [1.2 Target Function](#1-2-target-function)
    * [1.3 Implementing Gradient Descent](#1-3-implementing-gradient-descent)
    * [1.4 Choosing the Learning Rate](#1-4-choosing-the-learning-rate)
2. [Gradient Descent for Linear Regression](#2-gradient-descent-for-linear-regression)
3. [Feature Scaling](#3-feature-scaling)
4. [Gradient Descent for Multiple Linear Regression (Optional)](#4-multiple-linear-regression)
5. [Closed-Form Solution vs. Gradient Descent](#5-closed-form-vs-gradient-descent)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

## 1. Gradient Descent for Function Minimization <a id="1-gradient-descent-for-function-minimization"></a>

[Back to top](#contents)

In this section, we implement gradient descent and use it to find the minimum of a simple function of two variables - independent of any application in data science.

### 1.1 Plot Functions <a id="1-1-plot-functions"></a>

The following cell defines functions to visualize a function of two variables and the gradient descent process. Execute it to load the definitions to your kernel - you do not have to understand their implementation.

In [ ]:
def get_value_grid(x_min, x_max, y_min, y_max, z_func):
    xs = np.linspace(x_min, x_max, num=50, endpoint=True)
    ys = np.linspace(y_min, y_max, num=50, endpoint=True)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = z_func(X[i, j], Y[i, j])
    return X, Y, Z

def plot_func_3D(func, x_range, y_range):
    X, Y, Z = get_value_grid(x_range[0], x_range[1], y_range[0], y_range[1], func)
    fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
    ax.plot_surface(X, Y, Z, cmap=cm.rainbow, linewidth=0, antialiased=False)

def plot_func_2D_color_encoded(func, x_range, y_range, labels=('x', 'y'), log_scale=False):
    X, Y, Z = get_value_grid(x_range[0], x_range[1], y_range[0], y_range[1], func)
    plt.figure()
    plt.scatter(X, Y, c=np.log10(Z) if log_scale else Z, cmap=cm.rainbow)
    plt.colorbar(label='log10 of function value' if log_scale else 'function value')
    plt.xlabel(labels[0])
    plt.ylabel(labels[1])

def draw_descent_steps(xs, ys):
    plt.plot(xs, ys, color=(1, 1, 1), marker='.')
    plt.text(xs[0], ys[0], 1, color=(0, 0, 0))
    plt.text(xs[-1], ys[-1], len(xs), color=(0, 0, 0))

def plot_gradient_descent_process(xs, ys, target_fn, num_steps, x_true, y_true, labels=('x', 'y'), log_scale=False):
    xs = xs[:num_steps]
    ys = ys[:num_steps]
    x_margin = max(1.0, 0.1 * (np.max(xs) - np.min(xs)))
    y_margin = max(1.0, 0.1 * (np.max(ys) - np.min(ys)))
    plot_func_2D_color_encoded(target_fn, (np.min(xs) - x_margin, np.max(xs) + x_margin),
                               (np.min(ys) - y_margin, np.max(ys) + y_margin), labels, log_scale)
    draw_descent_steps(xs, ys)
    plt.scatter(x_true, y_true, marker='o', color=(0, 1, 0))

### 1.2 Target Function <a id="1-2-target-function"></a>

We want to find the minimum of the quadratic function ``f_quad`` of two variables using gradient descent.

In [ ]:
def f_quad(x, y):
    return (x * 2.5 - 3)**2 + (y * 1.7 + 1)**2

Implement the functions ``f_quad_dx`` and ``f_quad_dy``, which compute the partial derivatives of ``f_quad`` w.r.t. ``x`` and ``y``, respectively.

In [ ]:
def f_quad_dx(x, y):
    ### BEGIN SOLUTION
    return 2 * (x * 2.5 - 3) * 2.5
    ### END SOLUTION

def f_quad_dy(x, y):
    ### BEGIN SOLUTION
    return 2 * (y * 1.7 + 1) * 1.7
    ### END SOLUTION

In [ ]:
assert np.isclose(f_quad_dx(0, 0), -15) and np.isclose(f_quad_dx(2, 0), 10)
assert np.isclose(f_quad_dy(0, 0), 3.4) and np.isclose(f_quad_dy(0, 1), 9.18)

The following cells plot the function ``f_quad``: as a 3D surface and as a 2D plot, in which the color encodes the function value.

In [ ]:
plot_func_3D(f_quad, (-10, 10), (-10, 10))
plt.show()

In [ ]:
plot_func_2D_color_encoded(f_quad, (-10, 10), (-10, 10))
plt.show()

### 1.3 Implementing Gradient Descent <a id="1-3-implementing-gradient-descent"></a>

Implement the function ``gradient_descent_step_2d(x_0, y_0, target_fn, target_fn_dx, target_fn_dy, step_size)``,
which performs one step of gradient descent for a function ``target_fn`` with two arguments ``x, y``.

Inputs to the function are:
* the current position as two parameters ``x_0`` and ``y_0``,
* the function to be minimized ``target_fn``,
* the partial derivative of ``target_fn`` w.r.t. ``x`` as ``target_fn_dx``,
* the partial derivative of ``target_fn`` w.r.t. ``y`` as ``target_fn_dy``,
* the step size (learning rate) ``step_size``.

The function returns a tuple of two values ``x_1, y_1``: the position after one step of gradient descent.

In [ ]:
def gradient_descent_step_2d(x_0, y_0, target_fn, target_fn_dx, target_fn_dy, step_size):
    ### BEGIN SOLUTION
    dx = target_fn_dx(x_0, y_0)
    dy = target_fn_dy(x_0, y_0)
    x_1 = x_0 - step_size * dx
    y_1 = y_0 - step_size * dy
    return x_1, y_1
    ### END SOLUTION

The following cell contains a small test case for your implementation. The function and its derivatives are passed as ``lambda`` functions: ``lambda x, y: 2*x - y`` is a short way of defining a function with the arguments ``x, y``, which returns ``2*x - y``.

In [ ]:
x_1, y_1 = gradient_descent_step_2d(0, 1, lambda x, y: 2*x - y, lambda x, y: 2, lambda x, y: -1, 0.1)
assert np.isclose(x_1, -0.2)
assert np.isclose(y_1, 1.1)

Implement the function ``gradient_descent(x_0, y_0, target_fn, target_fn_dx, target_fn_dy, step_size, stop_eps, max_iter)``,
which performs multiple steps of gradient descent, starting at position ``x_0, y_0``.
Store the position after each iteration in two lists ``xs`` and ``ys``.
The gradient descent stops when the squared Euclidean distance between two succeeding positions is smaller than the threshold ``stop_eps``, or after ``max_iter`` steps, whichever comes first.
As each step is the learning rate times the gradient, a small step means a small gradient: this corresponds to the stopping criterion "gradient is very small" from the lecture.

The function returns the two lists ``xs`` and ``ys`` containing the positions after each iteration, including the starting position.

*Hints:*
* Use the previously implemented function ``gradient_descent_step_2d`` to perform a single step of gradient descent.
* In each iteration, compute the squared Euclidean distance $(x_{new} - x_{old})^2 + (y_{new} - y_{old})^2$ between the new and the previous position to check the stopping criterion.

In [ ]:
def gradient_descent(x_0, y_0, target_fn, target_fn_dx, target_fn_dy, step_size, stop_eps, max_iter):
    xs = [x_0]
    ys = [y_0]
    ### BEGIN SOLUTION
    for i in range(max_iter):
        x, y = gradient_descent_step_2d(xs[-1], ys[-1], target_fn, target_fn_dx, target_fn_dy, step_size)
        diff = (x - xs[-1]) ** 2 + (y - ys[-1]) ** 2
        xs.append(x)
        ys.append(y)
        if diff < stop_eps:
            break
    ### END SOLUTION
    return xs, ys

Again, the following cell contains a small test case for your implementation.

In [ ]:
xs, ys = gradient_descent(0, 1, lambda x, y: y**2 + 3, lambda x, y: 0, lambda x, y: 2*y, 0.1, 1e-6, 10)
assert len(xs) == 11
assert len(ys) == 11
assert np.std(np.array(xs)) < 1e-6
assert np.all(np.diff(ys) < 0)

### 1.4 Choosing the Learning Rate <a id="1-4-choosing-the-learning-rate"></a>

Find the minimum of ``f_quad`` with your function ``gradient_descent``.
Start at ``x_0 = 0, y_0 = 0`` and use a maximum of ``max_iter = 100`` iterations and the stopping threshold ``stop_eps = 1e-6``.

Try to find the learning rate which requires the smallest number of iterations to reach the minimum. Print the number of iterations.

In [ ]:
### BEGIN SOLUTION
xs, ys = gradient_descent(0, 0, f_quad, f_quad_dx, f_quad_dy, 0.1, 1e-6, 100)
print("Number of iterations:", len(xs) - 1)
### END SOLUTION

Use the function ``plot_gradient_descent_process(xs, ys, target_fn, num_steps, x_true, y_true)`` to visualize the gradient descent process.
Besides the positions and the target function, it expects the number of steps to plot and the coordinates ``x_true, y_true`` of the true minimum of ``f_quad``, which is marked in green.
Determine the true minimum analytically.

*Hint:* ``f_quad`` is a sum of two squared terms. Each term is minimal if it equals zero.

In [ ]:
### BEGIN SOLUTION
plot_gradient_descent_process(xs, ys, f_quad, 10, 3/2.5, -1/1.7)
### END SOLUTION
plt.show()

**Experiment:** In the next two cells, run gradient descent with a learning rate that is (1) too small and (2) too large. Visualize both processes.

In [ ]:
# Too small learning rate
### BEGIN SOLUTION
xs, ys = gradient_descent(0, 0, f_quad, f_quad_dx, f_quad_dy, 0.001, 1e-6, 100)
plot_gradient_descent_process(xs, ys, f_quad, 100, 3/2.5, -1/1.7)
### END SOLUTION
plt.show()

In [ ]:
# Too large learning rate
### BEGIN SOLUTION
xs, ys = gradient_descent(0, 0, f_quad, f_quad_dx, f_quad_dy, 0.3, 1e-6, 10)
plot_gradient_descent_process(xs, ys, f_quad, 5, 3/2.5, -1/1.7)
### END SOLUTION
plt.show()

#### Explanation

The minimum lies at $x = \frac{3}{2.5} = 1.2$ and $y = -\frac{1}{1.7} \approx -0.59$.

With a learning rate of about 0.1, the minimum is reached in less than 10 iterations. In the $x$ direction, the function is steeper than in the $y$ direction - a learning rate that is good for one direction is not optimal for the other.

* **Too small** (e.g., 0.001): the steps are tiny, after 100 iterations the minimum is still far away.
* **Too large** (e.g., 0.3): the steps overshoot the minimum in the steep $x$ direction, the positions jump back and forth with increasing distance - gradient descent **diverges**.

## 2. Gradient Descent for Linear Regression <a id="2-gradient-descent-for-linear-regression"></a>

[Back to top](#contents)

Now, we apply gradient descent to fit the production time model of the previous exercise: the production time $y$ as a linear function of the inverse cutting speed $x = \frac{1}{v}$:

$$
\hat{y} = w_0 + w_1 \cdot x
$$

Read ``laser_cutting_clean.csv`` from the directory ``4_data``, remove the parts with a missing ``ProductionTime`` and create the NumPy arrays ``x`` (inverse cutting speed) and ``y`` (production time).

In [ ]:
### BEGIN SOLUTION
laser = pd.read_csv('../4_data/laser_cutting_clean.csv')
laser_time = laser.dropna(subset=['ProductionTime'])
x = 1 / laser_time['CuttingSpeed_mm_s'].to_numpy()
y = laser_time['ProductionTime'].to_numpy()
### END SOLUTION

assert x.shape == y.shape == (len(laser_time),)

For comparison, the following cell computes the closed-form solution with the normal equation (see the previous exercise).

In [ ]:
X = np.column_stack([np.ones(len(x)), x])
w_closed = np.linalg.solve(X.T @ X, X.T @ y)
print(f'Closed-form solution: w0 = {w_closed[0]:.3f}, w1 = {w_closed[1]:.3f}')

In the lecture, gradient descent minimizes the residual sum of squares $S$ (RSS), i.e., the sum of squared errors of the previous exercise. Here, we use the **mean squared error** (MSE) of the model instead, which we already used for the RMSE in the previous exercise:

$$
MSE(w_0, w_1) = \frac{1}{n} \sum_{i=1}^{n} (y_i - w_0 - w_1 x_i)^2
$$

Its partial derivatives are the partial derivatives of $S$ from the lecture, divided by $n$:

$$
\frac{\partial MSE}{\partial w_0} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - w_0 - w_1 x_i), \qquad
\frac{\partial MSE}{\partial w_1} = -\frac{2}{n} \sum_{i=1}^{n} x_i \, (y_i - w_0 - w_1 x_i)
$$

Using the mean instead of the sum does not change the position of the minimum, but makes the choice of the learning rate independent of the number of samples.

Implement the three functions ``mse``, ``mse_dw0`` and ``mse_dw1``. Use NumPy operations on whole arrays instead of loops.

In [ ]:
def mse(w0, w1, x, y):
    ### BEGIN SOLUTION
    return np.mean((y - w0 - w1 * x) ** 2)
    ### END SOLUTION

def mse_dw0(w0, w1, x, y):
    ### BEGIN SOLUTION
    return -2 * np.mean(y - w0 - w1 * x)
    ### END SOLUTION

def mse_dw1(w0, w1, x, y):
    ### BEGIN SOLUTION
    return -2 * np.mean(x * (y - w0 - w1 * x))
    ### END SOLUTION

The following tests check your functions with a small example. At the closed-form solution, both partial derivatives must be zero (minimum).

In [ ]:
x_test, y_test = np.array([0.0, 1.0, 2.0]), np.array([1.0, 2.0, 4.0])
assert np.isclose(mse(0, 1, x_test, y_test), 2)
assert np.isclose(mse_dw0(0, 1, x_test, y_test), -8/3)
assert np.isclose(mse_dw1(0, 1, x_test, y_test), -10/3)
assert np.isclose(mse_dw0(w_closed[0], w_closed[1], x, y), 0, atol=1e-6)
assert np.isclose(mse_dw1(w_closed[0], w_closed[1], x, y), 0, atol=1e-6)

Our function ``gradient_descent`` from Section 1 expects functions with the two arguments of the position - here ``w0, w1``. The data ``x, y`` is fixed during the optimization.
The following cell creates such functions with ``lambda``:

In [ ]:
loss = lambda w0, w1: mse(w0, w1, x, y)
loss_dw0 = lambda w0, w1: mse_dw0(w0, w1, x, y)
loss_dw1 = lambda w0, w1: mse_dw1(w0, w1, x, y)

Fit the model with your function ``gradient_descent``: start at ``w0 = 0, w1 = 0`` with the learning rate ``0.5``, ``stop_eps = 1e-6`` and ``max_iter = 10000``.
Store the positions in ``w0s`` and ``w1s``.

Print the number of iterations, the final parameters and the MSE, and compare them to the closed-form solution.

In [ ]:
### BEGIN SOLUTION
w0s, w1s = gradient_descent(0, 0, loss, loss_dw0, loss_dw1, 0.5, 1e-6, 10000)
print(f'Iterations: {len(w0s) - 1}')
print(f'Gradient descent:     w0 = {w0s[-1]:.3f}, w1 = {w1s[-1]:.3f}, MSE = {loss(w0s[-1], w1s[-1]):.4f}')
print(f'Closed-form solution: w0 = {w_closed[0]:.3f}, w1 = {w_closed[1]:.3f}, MSE = {loss(w_closed[0], w_closed[1]):.4f}')
### END SOLUTION

Visualize the training process:
* Plot the MSE over the iterations. Use a logarithmic scale for the y-axis (``plt.yscale('log')``).
* Plot the path of the parameters with ``plot_gradient_descent_process()``. Pass ``labels=('w0', 'w1')`` and ``log_scale=True`` (the color then encodes the logarithm of the MSE).

In [ ]:
### BEGIN SOLUTION
plt.plot([loss(w0, w1) for w0, w1 in zip(w0s, w1s)])
plt.yscale('log')
plt.xlabel('Iteration')
plt.ylabel('MSE')
plt.show()

plot_gradient_descent_process(w0s, w1s, loss, len(w0s), w_closed[0], w_closed[1], labels=('w0', 'w1'), log_scale=True)
plt.show()
### END SOLUTION

**Experiment:** Repeat the gradient descent with the learning rates 0.1, 0.9 and 1.0 and print the number of iterations and the final parameters. What do you observe?

Why does gradient descent need so many iterations, and why does it stop before reaching the closed-form solution? Look at the path in the plot above.

In [ ]:
### BEGIN SOLUTION
for learning_rate in [0.1, 0.9, 1.0]:
    w0s_lr, w1s_lr = gradient_descent(0, 0, loss, loss_dw0, loss_dw1, learning_rate, 1e-6, 10000)
    print(f'learning rate {learning_rate}: {len(w0s_lr) - 1} iterations, w0 = {w0s_lr[-1]:.3f}, w1 = {w1s_lr[-1]:.3f}')
### END SOLUTION

#### Explanation

With the learning rate 0.5, gradient descent needs more than 2,500 iterations and stops close to, but not exactly at the closed-form solution. The MSE of both solutions is almost identical.

* With 0.1, the maximum number of iterations is reached before the stopping criterion; with 0.9, fewer iterations are needed.
* With 1.0, the parameters grow larger and larger: gradient descent **diverges**.

The path shows why: in the first iterations, the parameters quickly move into a long, narrow **valley** of the MSE. Along the valley, the MSE decreases only very slowly, so the gradient is small and the steps are tiny.
The learning rate cannot be increased to speed this up, as the steps across the valley would then overshoot (divergence).
When the steps become smaller than the threshold ``stop_eps``, gradient descent stops - although the minimum has not been reached yet. A small step size is not a guarantee for being at the minimum.

## 3. Feature Scaling <a id="3-feature-scaling"></a>

[Back to top](#contents)

The narrow valley results from the values of the feature: the inverse cutting speeds are small (between about 0.02 and 0.33) and far from zero compared to their spread.
Therefore, changes of $w_0$ and $w_1$ have very different effects on the MSE, and both parameters depend on each other.

A common remedy is to **standardize** the feature, i.e., to subtract its mean and divide by its standard deviation (compare the z-score in the EDA):

$$
z_i = \frac{x_i - \bar{x}}{s_x}
$$

Then, we fit the model $\hat{y} = u_0 + u_1 \cdot z$.

Create the standardized feature ``z`` and the corresponding functions ``loss_z``, ``loss_z_du0`` and ``loss_z_du1`` (analogous to Section 2).
Fit the model with the same settings as before (learning rate 0.5, start at ``0, 0``) and print the number of iterations.

In [ ]:
### BEGIN SOLUTION
x_mean, x_std = x.mean(), x.std()
z = (x - x_mean) / x_std
loss_z = lambda u0, u1: mse(u0, u1, z, y)
loss_z_du0 = lambda u0, u1: mse_dw0(u0, u1, z, y)
loss_z_du1 = lambda u0, u1: mse_dw1(u0, u1, z, y)

u0s, u1s = gradient_descent(0, 0, loss_z, loss_z_du0, loss_z_du1, 0.5, 1e-6, 10000)
print(f'Iterations: {len(u0s) - 1}, u0 = {u0s[-1]:.3f}, u1 = {u1s[-1]:.3f}')
### END SOLUTION

assert np.isclose(z.mean(), 0) and np.isclose(z.std(), 1)

The parameters $u_0, u_1$ refer to the standardized feature. Inserting the definition of $z$ into the model gives the parameters for the original feature:

$$
w_1 = \frac{u_1}{s_x}, \qquad w_0 = u_0 - u_1 \frac{\bar{x}}{s_x}
$$

Compute ``w0`` and ``w1`` and compare them with the closed-form solution. Plot the path of the parameters ``u0s, u1s`` with ``plot_gradient_descent_process()``; the minimum of the standardized problem is at the last position.

**Experiment:** Try the learning rates 0.1 and 0.9 as well.

In [ ]:
### BEGIN SOLUTION
w1_scaled = u1s[-1] / x_std
w0_scaled = u0s[-1] - u1s[-1] * x_mean / x_std
print(f'Gradient descent (scaled): w0 = {w0_scaled:.3f}, w1 = {w1_scaled:.3f}')
print(f'Closed-form solution:      w0 = {w_closed[0]:.3f}, w1 = {w_closed[1]:.3f}')

plot_gradient_descent_process(u0s, u1s, loss_z, len(u0s), u0s[-1], u1s[-1], labels=('u0', 'u1'), log_scale=True)
plt.show()

for learning_rate in [0.1, 0.9]:
    u0s_lr, u1s_lr = gradient_descent(0, 0, loss_z, loss_z_du0, loss_z_du1, learning_rate, 1e-6, 10000)
    print(f'learning rate {learning_rate}: {len(u0s_lr) - 1} iterations')
### END SOLUTION

#### Explanation

With the standardized feature, gradient descent needs only a few iterations (2 with the learning rate 0.5, about 40-50 with 0.1 or 0.9 instead of thousands) and finds the closed-form solution almost exactly.

* **Centering** (subtracting the mean) makes the two parameters independent of each other: $u_0$ is simply the mean production time.
* **Scaling** (dividing by the standard deviation) makes the MSE equally steep in both directions.

The valley turns into a round bowl (note the different scales of the two axes in the plot), and the negative gradient points directly to the minimum.
Feature scaling is important for all methods based on gradient descent, e.g., neural networks, and for many other machine learning methods (see later sessions).

## 4. Gradient Descent for Multiple Linear Regression (Optional) <a id="4-multiple-linear-regression"></a>

[Back to top](#contents)

**Optional:** With many parameters, it is impractical to implement one function per partial derivative. Using the feature matrix $X$ (first column of ones), all predictions are $X \mathbf{w}$ and all partial derivatives of the MSE can be computed at once as the **gradient vector**:

$$
\nabla MSE(\mathbf{w}) = -\frac{2}{n} X^\top (\mathbf{y} - X \mathbf{w})
$$

Implement the function ``gradient_descent_linear_regression(X, y, learning_rate, n_iterations)``. It starts with $\mathbf{w} = \mathbf{0}$, performs ``n_iterations`` steps of gradient descent and returns the final parameter vector ``w`` and a list with the MSE before each step.

Fit the burr height model for the steel parts from the previous exercise (without the gas supply failure) with standardized features. The data preparation is already given.
Use the learning rate 0.1 and 500 iterations, plot the MSE over the iterations and compare the parameters with the closed-form solution.

In [ ]:
features = ['Thickness_mm', 'CuttingSpeed_mm_s', 'LaserPower_W', 'GasPressure_bar']
steel = laser[(laser['Material'] == 'Steel') & (laser['GasPressure_bar'] > 4)].dropna(subset=features)
features_standardized = (steel[features] - steel[features].mean()) / steel[features].std()
X_steel = np.column_stack([np.ones(len(steel)), features_standardized.to_numpy()])
y_steel = steel['Gth_micro'].to_numpy()

In [ ]:
def gradient_descent_linear_regression(X, y, learning_rate, n_iterations):
    w = np.zeros(X.shape[1])
    mse_list = []
    ### BEGIN SOLUTION
    for _ in range(n_iterations):
        residuals = y - X @ w
        mse_list.append(np.mean(residuals ** 2))
        gradient = -2 / len(y) * X.T @ residuals
        w = w - learning_rate * gradient
    ### END SOLUTION
    return w, mse_list

In [ ]:
### BEGIN SOLUTION
w_gd, mse_list = gradient_descent_linear_regression(X_steel, y_steel, 0.1, 500)
w_steel_closed = np.linalg.solve(X_steel.T @ X_steel, X_steel.T @ y_steel)

plt.plot(mse_list)
plt.yscale('log')
plt.xlabel('Iteration')
plt.ylabel('MSE')
plt.show()

pd.DataFrame({'gradient descent': w_gd, 'closed form': w_steel_closed}, index=['Intercept'] + features).round(4)
### END SOLUTION

#### Explanation

After a few hundred iterations, gradient descent finds the closed-form solution. The same implementation works for any number of features - only the shape of $X$ changes.

The parameters now refer to the standardized features: they describe the change of the burr height per standard deviation of a feature.
This makes the effects of features with different units comparable: here, the cutting speed has the largest influence (about 3.8 µm per standard deviation), followed by the thickness (about 2.0 µm).

## 5. Closed-Form Solution vs. Gradient Descent <a id="5-closed-form-vs-gradient-descent"></a>

[Back to top](#contents)

Compare both approaches based on your experiences in this and the previous exercise: When would you use the closed-form solution, when gradient descent?

#### Explanation

| | Closed-form solution | Gradient descent |
|---|---|---|
| Result | exact minimum in one step | approximation, depends on learning rate, start and stopping criterion |
| Requirements | an explicit formula must exist (e.g., linear regression) | the gradient of the loss and a suitable learning rate |
| Effort | one linear equation system; expensive for very many features | many iterations; each one cheap, also for very large datasets (mini-batch variants use only a part of the data per step) |
| Pitfalls | singular matrix for perfectly collinear features | divergence, slow convergence without feature scaling |

For linear regression on our small datasets, the closed-form solution is the better choice. Gradient descent becomes indispensable for models without closed-form solution, e.g., neural networks - which are trained with the same principle as in this exercise.